# Sequence Tagging: A Tale of Two Approaches

**Target Audience:** Undergraduate Students  
**Prerequisites:** Basic Python, Introduction to Probability

## 1. The Big Question

Before we look at algorithms like HMMs or MaxEnt, ask yourself this single question:

> **“Do we want to learn how the data is PRODUCED, or only how to LABEL it correctly?”**

That single question separates the two main families of models in NLP:
1.  **Generative Models** (e.g., Hidden Markov Models)
2.  **Discriminative Models** (e.g., MaxEnt, CRFs)

## 2. The Movie Analogy

### 🎬 Generative Model = The Movie Director
Imagine a **Director**. They know how to create the entire scene from scratch.
*   They write the script.
*   They decide the characters' actions.
*   **Goal:** Make the *whole movie* (data) look realistic.

> **"If I know how the movie is made, I can replay it."**

### 🎯 Discriminative Model = The Movie Critic
Imagine a **Critic**. They don't know how to film a scene or direct actors. They just watch the final product and rate it.
*   They look at the screen.
*   They decide "Action", "Comedy", or "Drama".
*   **Goal:** Make the *label* correct.

> **"I don't care how it was made — just tell me what it is."**

## 3. The Math (Lightly)

Let's put this into probability terms without getting bogged down.

### Generative (HMM)
We try to maximize the probability of **both the words and the tags** existing together.

$$ \text{Maximize } P(\text{words}, \text{tags}) $$

**Ideally:** "We tune the model so the whole sentence + tags look most probable as a generated sequence."

### Discriminative (MaxEnt)
We try to maximize the probability of the **tags given the words**.

$$ \text{Maximize } P(\text{tags} \mid \text{words}) $$

**Ideally:** "We tune the model so the correct tag wins when we see this sentence."

### Example: "The bank is closed"
Let's look at the word **"bank"** in our starting example.

**HMM (Director):**
"I need to generate a sentence. If I just generated a Determinant ('The'), it is very likely (`Transition Prob`) that the next tag is a Noun. If the tag is Noun, it is somewhat likely (`Emission Prob`) that the word is 'bank'."

**MaxEnt (Critic):**
"I see the word 'bank'. I don't care about generating it. I just look at clues:
*   Clue 1: Is the previous word 'The'? Yes.
*   Clue 2: Does it end in '-ing'? No.
Conclusion: It's a Noun."


## 4. The Director: Hidden Markov Models (HMM)

The HMM is our Director. It follows a script based on probabilities.

### The "Script" Rules
1.  **Transition (The Plot Flow):** What tag comes next? (e.g., A Noun is often followed by a Verb).
    $$P(tag_i | tag_{i-1})$$
2.  **Emission (The Dialog):** Given a tag, what word is spoken? (e.g., If the tag is Noun, the word might be "Dog").
    $$P(word_i | tag_i)$$

### The Decoding Problem (Viterbi)
We have the movie (the words), but we lost the script notes (the tags). We need to figure out the most likely sequence of tags.

We use the **Viterbi Algorithm** (Dynamic Programming) to find the single best path through the potential script.

### 4.1 Learning from a Corpus (The Director's Research)

Instead of making up probabilities, let's learn them from a "Mini Corpus".

**Our Mini Corpus:**
1. "The bank is closed ."
2. "The bank is near ."
3. "I went to the bank ."
4. "The cat sits on the mat ."
5. "She runs fast ."
6. "Time flies like an arrow ."


In [6]:
from collections import defaultdict

# Mini Corpus: List of (word, tag) tuples
corpus = [
    [('The', 'DT'), ('bank', 'NN'), ('is', 'VBZ'), ('closed', 'VBN'), ('.', '.')],
    [('The', 'DT'), ('bank', 'NN'), ('is', 'VBZ'), ('near', 'RB'), ('.', '.')],
    [('I', 'PRP'), ('went', 'VBD'), ('to', 'TO'), ('the', 'DT'), ('bank', 'NN'), ('.', '.')],
    [('The', 'DT'), ('cat', 'NN'), ('sits', 'VBZ'), ('on', 'IN'), ('the', 'DT'), ('mat', 'NN'), ('.', '.')],
    [('She', 'PRP'), ('runs', 'VBZ'), ('fast', 'RB'), ('.', '.')],
    [('Time', 'NN'), ('flies', 'VBZ'), ('like', 'IN'), ('an', 'DT'), ('arrow', 'NN'), ('.', '.')]
]

# 1. Calculate Probabilities (Training the Director)
# TRANSITIONS: Count (tag_i | tag_i-1)
transitions = defaultdict(lambda: defaultdict(int))
tag_counts = defaultdict(int)

# EMISSIONS: Count (word_i | tag_i)
emissions = defaultdict(lambda: defaultdict(int))

for sentence in corpus:
    previous_tag = "<START>"
    for word, tag in sentence:
        transitions[previous_tag][tag] += 1
        emissions[tag][word] += 1
        tag_counts[tag] += 1
        previous_tag = tag

# Convert counts to probabilities
trans_prob = defaultdict(dict)
emit_prob = defaultdict(dict)

for prev_tag, next_tags in transitions.items():
    total = sum(next_tags.values())
    for tag, count in next_tags.items():
        trans_prob[prev_tag][tag] = count / total

for tag, words in emissions.items():
    total = sum(words.values())
    for word, count in words.items():
        emit_prob[tag][word] = count / total

print("Learned Transition Probabilities from 'DT':", dict(trans_prob['DT']))
print("Learned Emission Probabilities for 'NN':", dict(emit_prob['NN']))

# 2. Run Viterbi on "The bank is closed ."
# Using our learned probabilities

states = list(tag_counts.keys())
obs_sentence = ['The', 'bank', 'is', 'closed', 'at', 'five', '.']

# Prepare Viterbi inputs from our learned dicts
# Start prob is transition from <START>
start_p_learned = {st: trans_prob['<START>'].get(st, 0) for st in states}

print("\n--- Running Viterbi on 'The bank is closed at five .' ---")

def simple_viterbi_learned(obs, states, start_p, trans_p, emit_p):
    V = [{}]
    for st in states:
        # Handle unseen words smoothly (smoothing) - simple version: 1e-6
        emit = emit_p[st].get(obs[0], 1e-6)
        V[0][st] = {"prob": start_p[st] * emit, "prev": None}

    for t in range(1, len(obs)):
        V.append({})
        for st in states:
            max_tr_prob = 0
            prev_st_selected = None
            word = obs[t]
            emit = emit_p[st].get(word, 1e-6)

            for prev_st in states:
                trans = trans_p[prev_st].get(st, 0)
                # If previous path was impossible, skip
                if V[t-1][prev_st]["prob"] == 0: continue
                
                tr_prob = V[t-1][prev_st]["prob"] * trans * emit
                if tr_prob > max_tr_prob:
                    max_tr_prob = tr_prob
                    prev_st_selected = prev_st
            
            V[t][st] = {"prob": max_tr_prob, "prev": prev_st_selected}
            
    opt = []
    max_prob = 0
    best_st = None
    for st, data in V[-1].items():
        if data["prob"] > max_prob:
            max_prob = data["prob"]
            best_st = st
            
    if best_st:
        opt.append(best_st)
        for t in range(len(V) - 2, -1, -1):
             prev = V[t+1][best_st]["prev"]
             opt.insert(0, prev)
             best_st = prev

    print(f"Predicted Tags: {opt}")

simple_viterbi_learned(obs_sentence, states, start_p_learned, trans_prob, emit_prob)

Learned Transition Probabilities from 'DT': {'NN': 1.0}
Learned Emission Probabilities for 'NN': {'bank': 0.42857142857142855, 'cat': 0.14285714285714285, 'mat': 0.14285714285714285, 'Time': 0.14285714285714285, 'arrow': 0.14285714285714285}

--- Running Viterbi on 'The bank is closed at five .' ---
Predicted Tags: ['DT', 'NN', 'VBZ', 'IN', 'DT', 'NN', '.']


In [7]:
states

['DT', 'NN', 'VBZ', 'VBN', '.', 'RB', 'PRP', 'VBD', 'TO', 'IN']

In [8]:
import spacy
spacy.explain("VBD")

RuntimeError: generic_type: cannot initialize type "RpcBackendOptions": an object with that name is already defined

In [ ]:
dict(trans_prob)

{'<START>': {'DT': 0.5, 'PRP': 0.3333333333333333, 'NN': 0.16666666666666666},
 'DT': {'NN': 1.0},
 'NN': {'VBZ': 0.5714285714285714, '.': 0.42857142857142855},
 'VBZ': {'VBN': 0.2, 'RB': 0.4, 'IN': 0.4},
 'VBN': {'.': 1.0},
 'RB': {'.': 1.0},
 'PRP': {'VBD': 0.5, 'VBZ': 0.5},
 'VBD': {'TO': 1.0},
 'TO': {'DT': 1.0},
 'IN': {'DT': 1.0},
 '.': {}}

In [ ]:
dict(emit_prob)

{'DT': {'The': 0.5, 'the': 0.3333333333333333, 'an': 0.16666666666666666},
 'NN': {'bank': 0.42857142857142855,
  'cat': 0.14285714285714285,
  'mat': 0.14285714285714285,
  'Time': 0.14285714285714285,
  'arrow': 0.14285714285714285},
 'VBZ': {'is': 0.4, 'sits': 0.2, 'runs': 0.2, 'flies': 0.2},
 'VBN': {'closed': 1.0},
 '.': {'.': 1.0},
 'RB': {'near': 0.5, 'fast': 0.5},
 'PRP': {'I': 0.5, 'She': 0.5},
 'VBD': {'went': 1.0},
 'TO': {'to': 1.0},
 'IN': {'on': 0.5, 'like': 0.5}}

## 5. The Critic: Maximum Entropy (MaxEnt)

The MaxEnt model is our Critic. It doesn't care about the "script flow". It uses **Features** to make decisions.

### Why is it the Critic?
Because it can look at *anything* to make a decision, just like a critic can critique lighting, acting, or sound. 

HMMs are restricted (only previous tag). MaxEnt can ask:
*   "Does the word end in 'ing'?"
*   "Is the previous word 'The'?"
*   "Is the word capitalized?"

It combines all these features (critiques) to pick the winner.

$$ P(t|h) = \frac{e^{\sum \lambda_i f_i(h, t)}}{Z(h)} $$

(We pick the distribution that matches our features but is otherwise as random/unbiased (Maximum Entropy) as possible).

### 5.1 Training the Critic (MaxEnt from Corpus)

Let's train a MaxEnt classifier on the same Mini Corpus to tag the same sentence.

In [ ]:
import nltk
from nltk.classify import MaxentClassifier

# 1. Feature Extractor function
def get_features(sentence_words, index, previous_tag):
    word = sentence_words[index]
    return {
        'word': word,
        'is_capitalized': word[0].isupper(),
        'prev_tag': previous_tag,
        'prev_word': sentence_words[index-1] if index > 0 else '<START>'
    }

# 2. Prepare Training Data
# Format: [(features, label), ...]
train_data = []
for sentence in corpus:
    words = [w for w, t in sentence]
    prev_tag = "<START>"
    for i, (word, tag) in enumerate(sentence):
        feats = get_features(words, i, prev_tag)
        train_data.append((feats, tag))
        prev_tag = tag

# 3. Train Classifier (The Critic learns!)
# We set max_iter=10 for speed in this demo
print("Training MaxEnt Classifier...")
classifier = MaxentClassifier.train(train_data, max_iter=10)

# 4. Tag the sentence "The bank is closed ."
test_sentence = ['The', 'bank', 'is', 'closed', '.']
tags = []
prev_tag = "<START>"

print("\n--- MaxEnt Tagging on 'The bank is closed .' ---")
for i, word in enumerate(test_sentence):
    feats = get_features(test_sentence, i, prev_tag)
    tag = classifier.classify(feats)
    tags.append(tag)
    print(f"Word: {word:10} | Features: {str(feats):80} | Predicted Tag: {tag}")
    prev_tag = tag

print(f"\nFinal Sequence: {tags}")

Training MaxEnt Classifier...
  ==> Training (10 iterations)

      Iteration    Log Likelihood    Accuracy
      ---------------------------------------
             1          -2.30259        0.152
             2          -0.93279        0.909
             3          -0.63046        1.000
             4          -0.47705        1.000
             5          -0.38514        1.000
             6          -0.32384        1.000
             7          -0.27995        1.000
             8          -0.24690        1.000
             9          -0.22109        1.000
         Final          -0.20033        1.000

--- MaxEnt Tagging on 'The bank is closed .' ---
Word: The        | Features: {'word': 'The', 'is_capitalized': True, 'prev_tag': '<START>', 'prev_word': '<START>'} | Predicted Tag: DT
Word: bank       | Features: {'word': 'bank', 'is_capitalized': False, 'prev_tag': 'DT', 'prev_word': 'The'}  | Predicted Tag: NN
Word: is         | Features: {'word': 'is', 'is_capitalized': False, '

In [ ]:
train_data

[({'word': 'The',
   'is_capitalized': True,
   'prev_tag': '<START>',
   'prev_word': '<START>'},
  'DT'),
 ({'word': 'bank',
   'is_capitalized': False,
   'prev_tag': 'DT',
   'prev_word': 'The'},
  'NN'),
 ({'word': 'is',
   'is_capitalized': False,
   'prev_tag': 'NN',
   'prev_word': 'bank'},
  'VBZ'),
 ({'word': 'closed',
   'is_capitalized': False,
   'prev_tag': 'VBZ',
   'prev_word': 'is'},
  'VBN'),
 ({'word': '.',
   'is_capitalized': False,
   'prev_tag': 'VBN',
   'prev_word': 'closed'},
  '.'),
 ({'word': 'The',
   'is_capitalized': True,
   'prev_tag': '<START>',
   'prev_word': '<START>'},
  'DT'),
 ({'word': 'bank',
   'is_capitalized': False,
   'prev_tag': 'DT',
   'prev_word': 'The'},
  'NN'),
 ({'word': 'is',
   'is_capitalized': False,
   'prev_tag': 'NN',
   'prev_word': 'bank'},
  'VBZ'),
 ({'word': 'near',
   'is_capitalized': False,
   'prev_tag': 'VBZ',
   'prev_word': 'is'},
  'RB'),
 ({'word': '.',
   'is_capitalized': False,
   'prev_tag': 'RB',
   'prev_

## 6. The "Cheat Sheet" Table

| Question | Generative (HMM) | Discriminative (MaxEnt) |
| :--- | :--- | :--- |
| **Role** | Movie Director 🎬 | Movie Critic 🎯 |
| **What is learned?** | How data is made | How labels are chosen |
| **Likelihood** | $P(x, y)$ | $P(y \mid x)$ |
| **Can generate data?** | Yes | No |
| **Uses many features?** | Hard | Easy |

> **Summary:** Generative models maximize likelihood of data. Discriminative models maximize likelihood of correct labels.

## 7. Practical Example (NLTK & SpaCy)

Let's see the Critic in action. Most modern taggers (Standard NLTK, SpaCy) are Discriminative because accuracy is usually the priority.

In [ ]:
import nltk
import ssl

# Workaround for macOS SSL certificate issues
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

print("Downloading necessary NLTK data...")
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')
print("Download complete.")

sentence = "The bank is closed."

# 1. Tokenize
try:
    tokens = nltk.word_tokenize(sentence)
except LookupError:
    print("Retrying punkt download...")
    nltk.download('punkt')
    nltk.download('punkt_tab')
    tokens = nltk.word_tokenize(sentence)

# 2. POS Tagging (The Critic decides!)
# Note: The 'averaged_perceptron_tagger' is a discriminative model.
pos_tags = nltk.pos_tag(tokens)
print("\nPOS Tags:", pos_tags)

# 3. NER
# We also handle potential LookupError for chunking resources
try:
    ner_chunks = nltk.ne_chunk(pos_tags)
except LookupError:
    print("Retrying ne_chunker download...")
    nltk.download('maxent_ne_chunker')
    nltk.download('maxent_ne_chunker_tab')
    nltk.download('words')
    ner_chunks = nltk.ne_chunk(pos_tags)

print("\nNamed Entities:")
print(ner_chunks)

Download complete.

POS Tags: [('The', 'DT'), ('bank', 'NN'), ('is', 'VBZ'), ('closed', 'VBN'), ('.', '.')]


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/rajaramkankipati/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/rajaramkankipati/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/rajaramkankipati/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /Users/rajaramkankipati/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /Users/rajaramkankipati/nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     /Users/rajaramkankipati/nltk_data...
[nltk_data]   Package words is already up-to-dat


Named Entities:
(S The/DT bank/NN is/VBZ closed/VBN ./.)


### 7.1 Using SpaCy (Industrial Strength Critic)

NLTK is great for teaching, but **SpaCy** is often used in production because it's fast and accurate. It typically uses advanced models (like Neural Networks) that are also discriminative/critic-based.

In [ ]:
import spacy

print("Checking for SpaCy model 'en_core_web_sm'...")
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Model not found. Downloading 'en_core_web_sm'...")
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("Model loaded.")

# Process our example sentence
doc = nlp("The bank is closed at five.")

print("\n--- SpaCy POS Tags ---")
for token in doc:
    # text | coarse POS | fine-grained POS
    print(f"{token.text:10} | {token.pos_:6} | {token.tag_}")

RuntimeError: generic_type: cannot initialize type "RpcBackendOptions": an object with that name is already defined

In [9]:
# Process our example sentence
doc = nlp("A pilot likes flying planes.")

print("\n--- SpaCy POS Tags ---")
for token in doc:
    # text | coarse POS | fine-grained POS
    print(f"{token.text:10} | {token.pos_:6} | {token.tag_}")

NameError: name 'nlp' is not defined

In [ ]:
from spacy import displacy

In [ ]:
displacy.render(doc)